In [2]:
!pip install requests pandas tqdm


In [3]:
import requests
import pandas as pd
from tqdm import tqdm

BASE_URLS = [
    "https://gegevensmagazijn.tweedekamer.nl/OData/v4/2.0/Activiteit",
    "https://gegevensmagazijn.tweedekamer.nl/OData/v4/2.0/Besluit",
    "https://gegevensmagazijn.tweedekamer.nl/OData/v4/2.0/Persoon",
    "https://gegevensmagazijn.tweedekamer.nl/OData/v4/2.0/Vergadering",
    "https://gegevensmagazijn.tweedekamer.nl/OData/v4/2.0/Verslag",
    "https://gegevensmagazijn.tweedekamer.nl/OData/v4/2.0/Stemming",
    "https://gegevensmagazijn.tweedekamer.nl/OData/v4/2.0/Zaak",
]


In [4]:
def fetch_all_data(url, params=None):
    """
    Haalt alle resultaten op uit een OData endpoint (incl. pagination).
    """
    results = []
    
    while url:
        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json()
        
        results.extend(data.get("value", []))
        
        # volgende pagina indien aanwezig
        url = data.get("@odata.nextLink")
        params = None  # alleen eerste request heeft params
    
    return results


In [5]:
all_data = {}

for url in tqdm(BASE_URLS):
    name = url.split("/")[-1]
    print(f"Ophalen: {name}")
    
    data = fetch_all_data(
        url,
        params={
            "$format": "json;odata.metadata=none"  # kleinere responses
        }
    )
    
    df = pd.DataFrame(data)
    all_data[name] = df
    
    print(f"{name}: {len(df)} records")


  0%|          | 0/7 [00:00<?, ?it/s]

Ophalen: Activiteit


 14%|█▍        | 1/7 [01:30<09:00, 90.12s/it]

Activiteit: 95072 records
Ophalen: Besluit


 29%|██▊       | 2/7 [09:19<26:06, 313.20s/it]

...

Verslag: 22450 records
Ophalen: Stemming


 86%|████████▌ | 6/7 [20:38<04:31, 271.54s/it]

Stemming: 957152 records
Ophalen: Zaak


100%|██████████| 7/7 [25:07<00:00, 215.38s/it]

Zaak: 293928 records


In [6]:
all_data["Persoon"].head()


,Id,Nummer,Titels,Initialen,Tussenvoegsel,Achternaam,Voornamen,Roepnaam,Geslacht,Functie,...,Overlijdensdatum,Overlijdensplaats,Woonplaats,Land,ContentType,ContentLength,GewijzigdOp,ApiGewijzigdOp,Verwijderd,Fractielabel
0,fab499e2-93b6-4bba-8266-00014175f6a6,NaN,None,None,None,None,None,None,None,None,...,None,None,None,None,None,NaN,None,None,None,None
1,e55ca731-e1aa-44c0-a5f3-0008c23976f7,1034.0,None,J.J.,None,Atsma,Joop,Joop,man,Eerste Kamerlid,...,None,None,Surhuisterveen,NL,image/jpeg,27853.0,2023-08-29T13:09:45+02:00,2023-08-29T11:15:19.4179601Z,False,CDA
2,1c889902-6ede-427c-8682-000f683fffaa,2491.0,MA,M.,None,Agema,Marie-Fleur,Fleur,vrouw,Oud Kamerlid,...,None,None,'s-Gravenhage,NL,image/jpeg,456668.0,2024-07-02T12:11:06+02:00,2024-07-02T10:12:29.4687141Z,False,None
3,163423fd-6752-436a-9c5c-002ce57ae648,NaN,None,None,None,None,None,None,None,None,...,None,None,None,None,None,NaN,None,None,None,None
4,984df8b2-85d5-43cc-b1e8-0045cb0bf65f,1248.0,None,HA,None,Dambrink,None,None,man,Oud Kamerlid,...,1940-10-16,None,None,NL,None,NaN,2023-08-29T13:09:45+02:00,2023-08-29T11:16:24.2820522Z,False,None


In [7]:
for name, df in all_data.items():
    df.to_csv(f"{name}.csv", index=False)


In [1]:
import os 
import requests

BASE = "https://gegevensmagazijn.tweedekamer.nl/OData/v4/2.0/"
OUT = "transcripts"
os.makedirs(OUT, exist_ok=True)

url =  f"{BASE}/Verslag?$select=Id&$top=1000"
data = requests.get(url, timeout=60).json()

for row in data["value"]:
    vid = row["Id"]
    r = requests.get(f"{BASE}/Verslag({vid})/resource", timeout=120)

    with open(os.path.join(OUT, f"{vid}.xml"), "wb") as f:
        f.write(r.content)

KeyError: 'value'

In [2]:
import requests

BASE = "https://gegevensmagazijn.tweedekamer.nl/OData/v4/2.0"

url = f"{BASE}/Verslag?$select=Id&$top=1000"
r = requests.get(url, timeout=60)

print(r.status_code)
print(r.text[:500])   # show first 500 chars

data = r.json()
print(data.keys())


400
{"error":{"code":"","message":"The query specified in the URI is not valid. The limit of '250' for Top query has been exceeded. The value from the incoming request is '1000'.","details":[],"innererror":{"message":"The limit of '250' for Top query has been exceeded. The value from the incoming request is '1000'.","type":"Microsoft.OData.ODataException","stacktrace":"   at Microsoft.AspNetCore.OData.Query.Validator.TopQueryValidator.Validate(TopQueryOption topQueryOption, ODataValidationSettings v
dict_keys(['error'])


In [3]:
import requests

BASE = "https://gegevensmagazijn.tweedekamer.nl/OData/v4/2.0"

url = f"{BASE}/Verslag?$select=Id&$top=1000"
r = requests.get(url, timeout=60)

data = r.json()

if "value" not in data:
    print("API returned unexpected response:")
    print(data)
else:
    for row in data["value"]:
        vid = row["Id"]
        print(vid)

API returned unexpected response:
{'error': {'code': '', 'message': "The query specified in the URI is not valid. The limit of '250' for Top query has been exceeded. The value from the incoming request is '1000'.", 'details': [], 'innererror': {'message': "The limit of '250' for Top query has been exceeded. The value from the incoming request is '1000'.", 'type': 'Microsoft.OData.ODataException', 'stacktrace': '   at Microsoft.AspNetCore.OData.Query.Validator.TopQueryValidator.Validate(TopQueryOption topQueryOption, ODataValidationSettings validationSettings)\r\n   at Microsoft.AspNetCore.OData.Query.Validator.ODataQueryValidator.Validate(ODataQueryOptions options, ODataValidationSettings validationSettings)\r\n   at Microsoft.AspNetCore.OData.Query.EnableQueryAttribute.ValidateQuery(HttpRequest request, ODataQueryOptions queryOptions)\r\n   at Microsoft.AspNetCore.OData.Query.EnableQueryAttribute.OnActionExecuting(ActionExecutingContext actionExecutingContext)'}}}


In [4]:
import os
import time
import requests

BASE = "https://gegevensmagazijn.tweedekamer.nl/OData/v4/2.0"
OUT = "transcripts"
os.makedirs(OUT, exist_ok=True)

# 1) get 250 verslag ids
url = f"{BASE}/Verslag?$select=Id&$top=250&$skip=0"
data = requests.get(url, timeout=60).json()

for row in data["value"]:
    vid = row["Id"]

    # 2) download transcript xml for each verslag
    r = requests.get(f"{BASE}/Verslag({vid})/resource", timeout=120)

    with open(os.path.join(OUT, f"{vid}.xml"), "wb") as f:
        f.write(r.content)

    time.sleep(0.1)